In [1]:
import sys
import os

# Define paths dynamically
base_dir_str: str = os.path.abspath("")  
data_rel_path_str: str = os.path.join("..", "data") 
data_abs_path_str: str = os.path.normpath(os.path.join(base_dir_str, data_rel_path_str))

# Add the absolute path to sys.path to ensure module imports work correctly
if data_abs_path_str not in sys.path:
    sys.path.append(data_abs_path_str)

# Import your custom dataset
try:
    import dataset
    dataset_imported_bol: bool = True
except ImportError as e:
    print(f"Warning: Could not import dataset module from {data_abs_path_str}. Error: {e}")
    dataset_imported_bol: bool = False



In [2]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from typing import List

def get_recursive_chunks_list(
    input_str: str,
    chunk_size_int: int = 500,
    chunk_overlap_int: int = 50
) -> List[str]:
    """
    Splits a raw string into chunks using a recursive character approach
    to preserve semantic structure (paragraphs, then sentences).

    Args:
        input_str (str): The raw text data to be processed.
        chunk_size_int (int): The maximum number of characters per chunk.
        chunk_overlap_int (int): The number of characters to overlap between chunks.

    Returns:
        List[str]: A list of semantically grouped text chunks.
    """
    # Define the hierarchical separators
    recursive_splitter_obj = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size_int,
        chunk_overlap=chunk_overlap_int,
        separators=["\n\n", "\n", " ", ""]
    )

    chunks_list = recursive_splitter_obj.split_text(input_str)
    return chunks_list

# Sample data representing a structured document
raw_data_str = dataset.test

processed_chunks_list = get_recursive_chunks_list(raw_data_str)

print(f"Total chunks created: {len(processed_chunks_list)}")
print(f"First chunk preview: {processed_chunks_list[0]}")

c:\SJ\SD_tasks\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Total chunks created: 76
First chunk preview: Abstract
Background
Cancers arise from multiple acquired mutations, which presumably occur over many years. Early stages in cancer development might be present years before cancers become clinically apparent.
Methods


In [3]:
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

def create_recursive_vector_db(chunks_list: List[str]) -> Chroma:
    """
    Converts recursive chunks into embeddings and saves them to a vector store.

    Args:
        chunks_list (List[str]): List of processed text chunks.

    Returns:
        Chroma: The Chroma vector database object.
    """
    # Using a robust open-source embedding model
    embeddings_model_obj = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

    vector_db_obj = Chroma.from_texts(
        texts=chunks_list,
        embedding=embeddings_model_obj,
        collection_name="recursive_rag_collection"
    )

    return vector_db_obj

recursive_db_obj = create_recursive_vector_db(processed_chunks_list)

C:\Users\sjoshi06\AppData\Local\Temp\ipykernel_20132\3568481119.py:15: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings_model_obj = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 525.71it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [4]:
def retrieve_semantic_context(
    query_str: str,
    vector_db: Chroma,
    k_int: int = 3
) -> List[str]:
    """
    Retrieves the most semantically relevant chunks for a user query.

    Args:
        query_str (str): The user query.
        vector_db (Chroma): The vector database.
        k_int (int): Number of chunks to retrieve.

    Returns:
        List[str]: Top K relevant text chunks.
    """
    # Perform the similarity search
    retrieved_docs_list = vector_db.similarity_search(query_str, k=k_int)

    # Extract the text content from documents
    context_content_list = [doc_obj.page_content for doc_obj in retrieved_docs_list]
    return context_content_list

# Test Retrieval
user_question_str = "What is clonal hematopoiesis and how common is it in elderly individuals?"
results_list = retrieve_semantic_context(user_question_str, recursive_db_obj)

for idx_int, text_str in enumerate(results_list):
    print(f"Result {idx_int + 1}:\n{text_str}\n{'-'*20}")

Result 1:
Clonal Hematopoiesis and Advancing Age
--------------------
Result 2:
clonal hematopoiesis with unknown drivers) was more strongly age-dependent, occurring in only 0.3% of persons younger than 50 years of age but in 4.6% of those older than 65 years of age (Figure 2D). This age trajectory is similar to that of clonal hematopoiesis with candidate drivers.
--------------------
Result 3:
Detectable clonal hematopoiesis with candidate drivers was rare among younger participants (occurring in 0.7% of participants younger than 50 years of age) but much more common among older participants (occurring in 5.7% of participants older than 65 years of age) (Figure 2D). Reflecting this relationship, participants having clonal hematopoiesis with candidate drivers were, on average, older than those without detectable putative somatic mutations (mean age, 64 years vs. 55 years; P<0.001);
--------------------
